# SGLang + ASH-KV E2E Validation (MI300X Native)
This notebook is explicitly designed for the **AMD Pre-configured Cloud Image** containing:
- **ROCm**: 7.0.0
- **SGLang**: 0.5.9

It tests the ASH-KV integration using **real Triton kernels (compiled for AMD)** and **real PyTorch ROCm memory allocations** on your MI300X GPU.


In [ ]:
# 1. Setup ASH-KV inside the pre-configured environment
import sys
import os

# Ensure the kernel can find ASH-KV. If you cloned it in a specific directory, update this path:
repo_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_path not in sys.path:
    sys.path.append(repo_path)

import torch
import time
import matplotlib.pyplot as plt

from ashkv.contracts import PageTable
from ashkv.sglang_integration.allocator import SGLangShadowAllocator
from ashkv.sglang_integration.hooks import SGLangHooks
from ashkv.sglang_integration.patcher import apply_radix_cache_patches

# Import SGLang's internal RadixCache (pre-installed v0.5.9)
from sglang.srt.managers.radix_cache import RadixCache

assert torch.cuda.is_available(), "ROCm PyTorch is not detecting the GPU!"
print(f"Detected GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# 2. Memory Pool Mock for Controlled Testing
# We mock the allocator to artificially force an OOM condition at 2000 tokens
class MockTokenToKVPool:
    def __init__(self, capacity_tokens: int):
        self.capacity_tokens = capacity_tokens
        self.allocated_tokens = 0

    def allocate(self, num_tokens: int):
        if self.allocated_tokens + num_tokens > self.capacity_tokens:
            return None # OOM
        self.allocated_tokens += num_tokens
        return torch.arange(num_tokens) # Dummy physical slot indices

    def free(self, indices: torch.Tensor):
        if indices is not None:
            self.allocated_tokens -= len(indices)


In [ ]:
# 3. Patching SGLang's Real RadixCache (v0.5.9)
pt = PageTable()
# Shadow allocator: 100 MB budget on MI300X VRAM
shadow_alloc = SGLangShadowAllocator(max_bytes=100 * 1024 * 1024)
class DummyConfig:
    num_hidden_layers = 32
hooks = SGLangHooks(pt, shadow_alloc, DummyConfig())
pool = MockTokenToKVPool(capacity_tokens=2000)

# Create a Real PyTorch ROCm tensor to act as the KV Cache
# 32 layers, max 2000 tokens, 128 hidden dim
sglang_kv_cache = torch.zeros((32, 2000, 128), dtype=torch.bfloat16, device="cuda")

# Apply the Monkey Patches to SGLang's classes!
apply_radix_cache_patches(hooks, sglang_kv_cache, pool)
print("ASH-KV successfully hooked into SGLang RadixCache.")


In [ ]:
# 4. Load Testing & Validation
# Instantiate the Real SGLang RadixCache
cache = RadixCache(req_capacity=2000)

history_bf16 = []
history_int8 = []

def track_memory():
    history_bf16.append(pool.allocated_tokens)
    history_int8.append(shadow_alloc.allocated_bytes)

print("--- STARTING SGLANG HARDWARE SIMULATION ON MI300X ---")

print("\n[Tick 1] Generating System Prompt (1000 tokens)")
sys_node = cache.insert(tuple(range(1000)), value=1000) 
sys_node.kv_indices = pool.allocate(1000)
track_memory()

user_nodes = []
for i in range(5):
    print(f"\n[Tick 2.{i}] User {i} connecting and generating (400 tokens)")
    
    indices = pool.allocate(400)
    if indices is None:
        print("  [SGLang] Pool Full! Forcing SGLang RadixCache to Evict...")
        # Force SGLang's REAL evict() method to run. 
        # Our ASH-KV hooks will intercept, triggering REAL Triton Kernels on the MI300X!
        freed = cache.evict(400, evict_callback=lambda idx: pool.free(idx))
        if freed < 400:
            raise RuntimeError("Out of Memory! ASH-KV Hooks failed to free enough space.")
        indices = pool.allocate(400)
        
    u_node = cache.insert(tuple(range(1000)) + tuple(range(1000 + i*400, 1400 + i*400)), value=400)
    u_node.kv_indices = indices
    user_nodes.append(u_node)
    
    track_memory()
    time.sleep(0.01)

print("\n[Tick 3] User 0 returns to their chat (Prefix Hit)")
# SGLang match_prefix triggers the promote_hook in our patch!
matched_node, _, _ = cache.match_prefix(tuple(range(1000)) + tuple(range(1000, 1400)))
track_memory()

# Visualization
int8_token_equivalent = [b / 4096 for b in history_int8]

plt.figure(figsize=(10, 6))
plt.fill_between(range(len(history_bf16)), history_bf16, label="SGLang Physical VRAM (BF16)", alpha=0.7, color='blue')
plt.fill_between(range(len(int8_token_equivalent)), [b + i for b, i in zip(history_bf16, int8_token_equivalent)], history_bf16, label="ASH-KV Shadow VRAM (INT8)", alpha=0.7, color='orange')
plt.axhline(y=2000, color='r', linestyle='--', label="Physical VRAM Limit (2000 tokens)")

plt.title("ASH-KV E2E Hardware Validation (MI300X)")
plt.xlabel("Simulation Ticks (Requests)")
plt.ylabel("Stored Tokens (Equivalent)")
plt.legend(loc="upper left")
plt.grid(True, alpha=0.3)
plt.show()
